# Real World Dataset Analysis: Palmer Penguins

**Objective:** Explore a real world dataset, clean and preprocess it, perform exploratory data analysis, create visualizations, and present findings.

The Palmer Penguins dataset contains measurements of 344 penguins from three species observed in the Palmer Archipelago, Antarctica. The data includes species, island, bill measurements, flipper length, body mass, sex, and year.

**Important:** This notebook downloads the original CSV dataset from its public source when you run the first code cell. An internet connection is required for this step.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_URL = "https://huggingface.co/datasets/SIH/palmer-penguins/resolve/main/penguins.csv"
DATA_FILE = Path("penguins.csv")

if not DATA_FILE.exists():
    df = pd.read_csv(DATA_URL)
    df.to_csv(DATA_FILE, index=False)
else:
    df = pd.read_csv(DATA_FILE)

print("Dataset loaded successfully.")
print("Rows and columns:", df.shape)
display(df.head())


## 1. Understand the dataset

Inspect the column names, data types, dataset size, and summary statistics. Each row represents one penguin observation.


In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types and non missing values:")
df.info()

print("\nSummary statistics:")
display(df.describe(include="all").T)


## 2. Check data quality

Count missing values and exact duplicate rows. The source dataset contains missing observations, so checking these values is an important part of the analysis.


In [ ]:
print("Missing values in each column:")
display(df.isnull().sum().to_frame("missing_values"))

print("Number of exact duplicate rows:", df.duplicated().sum())


## 3. Clean and preprocess the data

Remove exact duplicate rows. Fill missing numeric values with the median of their column. Fill missing categorical values with the most common category. Median imputation is less affected by extreme values than the mean.


In [ ]:
clean_df = df.drop_duplicates().copy()

numeric_cols = clean_df.select_dtypes(include=np.number).columns
categorical_cols = clean_df.select_dtypes(include=["object", "category"]).columns

for col in numeric_cols:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

for col in categorical_cols:
    if clean_df[col].isna().any():
        clean_df[col] = clean_df[col].fillna(clean_df[col].mode().iloc[0])

print("Original shape:", df.shape)
print("Cleaned shape:", clean_df.shape)
print("\nMissing values after cleaning:")
display(clean_df.isnull().sum().to_frame("missing_values"))


## 4. Explore species and island distribution

Count penguins by species and island. Compare average body mass and flipper length for each species.


In [ ]:
print("Penguin count by species:")
display(clean_df["species"].value_counts().to_frame("count"))

print("Penguin count by island:")
display(clean_df["island"].value_counts().to_frame("count"))

species_summary = clean_df.groupby("species")[["body_mass_g", "flipper_length_mm", "bill_length_mm", "bill_depth_mm"]].mean().round(2)
display(species_summary)


## 5. Visualize the data

The charts below show the number of observations by species, average body mass by species, the relationship between flipper length and body mass, and the distribution of body mass.


In [ ]:
# Chart 1: species counts
plt.figure(figsize=(7, 4))
clean_df["species"].value_counts().plot(kind="bar")
plt.title("Number of Penguins by Species")
plt.xlabel("Species")
plt.ylabel("Number of penguins")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Chart 2: average body mass by species
plt.figure(figsize=(7, 4))
clean_df.groupby("species")["body_mass_g"].mean().plot(kind="bar")
plt.title("Average Body Mass by Species")
plt.xlabel("Species")
plt.ylabel("Average body mass (g)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Chart 3: flipper length and body mass
plt.figure(figsize=(7, 5))
for species, group in clean_df.groupby("species"):
    plt.scatter(group["flipper_length_mm"], group["body_mass_g"], label=species, alpha=0.7)
plt.title("Flipper Length and Body Mass")
plt.xlabel("Flipper length (mm)")
plt.ylabel("Body mass (g)")
plt.legend()
plt.tight_layout()
plt.show()

# Chart 4: body mass distribution
plt.figure(figsize=(7, 4))
plt.hist(clean_df["body_mass_g"], bins=20, edgecolor="black")
plt.title("Distribution of Penguin Body Mass")
plt.xlabel("Body mass (g)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## 6. Correlation and basic statistical analysis

Correlation measures the strength and direction of a linear relationship. It does not prove that one measurement causes another.


In [ ]:
numeric_data = clean_df.select_dtypes(include=np.number)
correlation = numeric_data.corr(numeric_only=True)
display(correlation.round(2))

print("Correlation between flipper length and body mass:",
      round(clean_df["flipper_length_mm"].corr(clean_df["body_mass_g"]), 3))

print("\nAverage measurements by species:")
display(clean_df.groupby("species")[["body_mass_g", "flipper_length_mm"]].agg(["mean", "median", "min", "max"]).round(2))


## 7. Identify possible outliers using the IQR method

The interquartile range (IQR) method flags unusually low or high numeric values for further review. A flagged value is not automatically an error, especially in biological data where natural variation is expected.


In [ ]:
for col in ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]:
    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((clean_df[col] < lower) | (clean_df[col] > upper)).sum()
    print(f"{col}: {count} possible outlier(s) using IQR")


## 8. Save the cleaned dataset and present findings

The cleaned CSV can be used for further analysis. Review the printed values and charts, then write a short conclusion based on the actual output produced on your computer.


In [ ]:
clean_df.to_csv("penguins_cleaned.csv", index=False)
print("Cleaned dataset saved as penguins_cleaned.csv")

print("\nKey findings to review:")
print("1. Species counts show how observations are distributed among the three species.")
print("2. Species averages help compare body mass, flipper length, and bill measurements.")
print("3. The scatter plot and correlation value show whether flipper length and body mass tend to change together.")
print("4. Missing values were handled using median and mode imputation.")
print("5. IQR flags possible outliers for review; they should not be removed without a valid reason.")


## Conclusion

This project demonstrates a complete beginner friendly data analysis workflow using a real world biological dataset. It includes data loading, inspection, missing value handling, duplicate checking, summary statistics, grouping, visualization, correlation analysis, and outlier review.

The analysis can show differences among penguin species and relationships between physical measurements. These are descriptive findings from the available observations. Correlation does not establish causation, and imputed values should be considered when interpreting the results.
